In [ ]:
import pickle

import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data
import matplotlib.pylab as pylab
import gc
import quantus
from tqdm import tqdm
from captum.attr import GradientShap

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
params = {'axes.labelsize': 'x-small',
         'axes.titlesize':'x-small',
         'xtick.labelsize':'x-small',
         'ytick.labelsize':'x-small'}
pylab.rcParams.update(params)

General Goal of this Notebook:\
Check if removing outliers in variance (top 5%) has effect on the topoplots when we group by uncertainty.\
This is done because in uncertainty_stat_analysis.ipynb it was observed that while the majority of variances in trials are <=1, there are some extremely high outliers for some trials, especially for some of the subjects.\
Also I want to check whether aggregation over the time_point and trial dimension with mean leads to vastly different result that only aggregation over the time_point dimension with mean and over the trial dimension linearly, via a top-k approach.\
The idea is that with a top-k approach, every trial is equally important, reducing the influence of potential outlier trials

Note not all plots included, due to github size restrictions. Therefore subselected (to me) most interesting plots. Also is the reason why plots are so small :(

# Boilerplate code

removed 25 due to NaNs

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += k-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(explanations, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = explanations.squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict, k)
    return channel_points_dict

def get_top_k_weighted(explanations, ch_names, k, update_func, index_group=np.array([]), groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if index_group.any() == False:
        if groupby_indices.any():
            data = explanations.squeeze()[groupby_indices]
        else:
            data = explanations.squeeze()
    else:
        if groupby_indices.any():
            data = explanations.squeeze()[index_group&groupby_indices]
        else:
            data = explanations.squeeze()[index_group]

    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return channel_point_dicts_over_time
    

def get_top_k_weighted_individual(explanations, ch_names, k, update_func, index_group=np.array([]), groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if index_group.any() == False:
        if groupby_indices.any():
            data = explanations.squeeze()[groupby_indices]
        else:
            data = explanations.squeeze()
    else:
        if groupby_indices.any():
            data = explanations.squeeze()[index_group&groupby_indices]
        else:
            data = explanations.squeeze()[index_group]

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]

    return channel_point_dicts_individual_trials


def top_k_groupby(explanations, groupby_dict, ch_names, k, update_func, index_group = np.array([]), top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(explanations, ch_names, k, update_func, index_group=index_group, groupby_indices=groupby_indices)
    
    return groupby_channel_points_dict
 


In [ ]:
def top_k(explanations, ch_names, k, update_func, top_k_func=get_top_k_weighted, index_group=np.array([])):
    return top_k_func(explanations, ch_names, k, update_func, index_group=index_group)

In [ ]:
def get_and_plot_topomap_groupby(explanations, groupby_dict, aggregation_func, info, indices=np.array([]), subject_idx=2, aggregation_func_name="mean", groupby_name="uncertainty", save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    # as the outputs of the XAI approaches come in the form trials x channels x timepoints, some form of summary stat needs to be applied.
    # this function allows for passing only a single trial and computing the mean over the timepoint dim,
    # or passing all trials and aggregating over trial and timepoints dim
    # Note: If only a single trial is passed, it needs to be passed in the shape 1 x channel x timepoints
    ncols = len(groupby_dict.keys())
    
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(6,2))
    if indices.any() == False:
        fig.suptitle(f"subject: {subject_idx}, aggregation: {aggregation_func_name}")
    else:
        fig.suptitle(f"subject: {subject_idx}, aggregation: {aggregation_func_name}, indices: {np.where(indices==True)[0][0]} : {(np.where(indices==True)[0][-1])}")
    for idx, groupby_key in enumerate(groupby_dict.keys()):

        #fig, ax = plt.subplots(figsize=(6,6))
        data = explanations.squeeze()
        if indices.any() == False:
            indices = np.ones(data.shape[0], dtype=bool)
        #print(aggregation_func(data[groupby_dict[groupby_key]], **kwargs))
        axs[idx].set_title(f"{groupby_name}: {groupby_key}, N: {len(data[groupby_dict[groupby_key]&indices])}",y=0.9)
        mne.viz.plot_topomap(aggregation_func(data[groupby_dict[groupby_key]&indices], **kwargs), info, axes=axs[idx], show=False)

        dir_path = f"{save_path}/{aggregation_func_name}"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/GradShap_{aggregation_func_name}_{groupby_name}_subject_{subject_idx}_topoplot.png")

In [ ]:
def get_and_plot_topomap(explanations, aggregation_func, info, indices=np.array([]), subject_idx=2, aggregation_func_name="mean", save_path="refactor_test/groupby/uncertainty/topoplots/", ax=None, **kwargs):
    # as the outputs of the XAI approaches come in the form trials x channels x timepoints, some form of summary stat needs to be applied.
    # this function allows for passing only a single trial and computing the mean over the timepoint dim,
    # or passing all trials and aggregating over trial and timepoints dim
    # Note: If only a single trial is passed, it needs to be passed in the shape 1 x channel x timepoints

    data = explanations.squeeze()
    if indices.any() == False:
        indices = np.ones(data.shape[0], dtype=bool)

    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
        fig.suptitle(f"subject: {subject_idx}, aggregation: {aggregation_func_name}")
        save_fig = True
    else:
        save_fig = False

    mne.viz.plot_topomap(aggregation_func(data[indices], **kwargs), info, axes=ax, show=False)

    if save_fig:
        dir_path = f"{save_path}/{aggregation_func_name}"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)
        fig.savefig(f"{dir_path}/time_GradShap_{aggregation_func_name}_subject_{subject_idx}_topoplot.png")


In [ ]:
def get_all_epochs(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    return all_epochs[150:], ch_names

# Call the function and get the first 150 epochs


In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
def compute_predictions_and_uncertainties(all_epochs, cfg, device, subject_index):
    pred_label_original = np.zeros((all_epochs.shape[0]))
    uncertainties_original = np.zeros((all_epochs.shape[0]))
    input_shape_st = (60, 900)
    
    for i in tqdm(range(0, len(all_epochs))):
        start_index = i + 100
        inputs = torch.from_numpy(all_epochs[i])
        inputs = inputs.to(device).float()
        inputs = inputs.unsqueeze(0)

        model = load_model(cfg, start_index=start_index, subject_index=subject_index)
        pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
        var = torch.exp(log_var)
        pred_label_original[i] = pred_mean.cpu().detach().numpy()
        uncertainties_original[i] = var.cpu().detach().numpy()
    
    return pred_label_original, uncertainties_original

# Usage
#cfg = load_config()
#pred_label_original, uncertainties_original = compute_predictions_and_uncertainties(all_epochs, cfg, device, subject_index)

In [ ]:

def gradshap_explainer(
    model, inputs, targets, abs=False, normalise=False, *args, **kwargs
) -> np.array:
    """Wrapper aorund captum's GradShap implementation."""
    

    gc.collect()
    torch.cuda.empty_cache()

    # Set model in evaluate mode.
    model.to(kwargs.get("device", None))
    model.eval()

    inputs = torch.from_numpy(inputs)
    inputs = inputs.to(kwargs.get("device", None)).float()


    baselines = torch.zeros_like(inputs).to(kwargs.get("device", None)).float()
    gs = GradientShap(model)
    explanation = (
        gs
        .attribute(inputs=inputs, target=targets, baselines=baselines)
    ).cpu().data

    gc.collect()
    torch.cuda.empty_cache()

    if normalise:
        explanation = quantus.normalise_func.normalise_by_negative(explanation)

    if isinstance(explanation, torch.Tensor):
        if explanation.requires_grad:
            return explanation.cpu().detach().numpy()
        return explanation.cpu().numpy()

    return explanation

In [ ]:

def compute_gradshap(all_epochs, subject_index=2):
    a_batch_gradshap = np.zeros_like(all_epochs)

    for i in tqdm(range(0, len(all_epochs))):
        start_index = i + 100
        x_batch = all_epochs[i]
        x_batch = x_batch[np.newaxis, :, :]
        cfg = load_config()
        model = load_model(cfg, start_index=start_index, subject_index=subject_index)
        a_batch_gradshap[i] = gradshap_explainer(model=model.cpu(), 
                                                  inputs=x_batch,
                                                  targets=0, 
                                                  **{"device": device})

        gc.collect()
        torch.cuda.empty_cache()
    
    return a_batch_gradshap

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

# compute predictions uncertainties and explanations for each subject

In [ ]:
# Function to load data, get predictions, uncertainties, and explanations for each subject
def process_subject_data(subject_indices):
    results = {}
    for subject_index in tqdm(subject_indices):
        print(f"Processing subject {subject_index}...")
        
        # Load data
        all_epochs, ch_names = get_all_epochs(subject_index=subject_index)
        
        # Compute predictions and uncertainties
        cfg = load_config()
        pred_label_original, uncertainties_original = compute_predictions_and_uncertainties(all_epochs, cfg, device, subject_index)
        
        # Compute explanations using GradShap
        explanations = compute_gradshap(all_epochs, subject_index=subject_index)
        
        # Save results
        results[subject_index] = {
            'predictions': pred_label_original,
            'uncertainties': uncertainties_original,
            'explanations': explanations
        }
        
        # Save to disk
        np.save(f'subject_{subject_index}_predictions.npy', pred_label_original, allow_pickle=True)

        
        print(f"Finished processing subject {subject_index}.")
    
    return results

# List of subject indices to process
#subject_indices = [52, 55, 56, 57, 60, 62, 67, 69, 72, 73, 79, 80, 86, 88, 92, 102]

# Process data for each subject
cfg = load_config()
subject_indices = cfg.dataset.test_subject_indices
#results = process_subject_data(subject_indices)

In [ ]:
def load_all_subject_results(subject_indices):
    results = {}
    for subject_index in subject_indices:
        results[subject_index]  = np.load(f'subject_{subject_index}_results.pkl', allow_pickle=True)
    return results

# List of subject indices to load
cfg = load_config()
subject_indices = cfg.dataset.test_subject_indices

# Load results for each subject
subjects_dicts = load_all_subject_results(subject_indices)

In [ ]:
def get_fixed_median(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    return fixed_median, ch_names

In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    fixed_median, ch_names = get_fixed_median(subject_index=subject_index)
    subjects_dicts[subject_index]["pred_binary_label_fixed"] = np.where(subjects_dicts[subject_index]['predictions'] > fixed_median, 1, 0)

In [ ]:
cwd = os.getcwd()
file_path = os.path.join(cwd, "../data/subject_007_preprocessed_combined_py.fif")
epochs = mne.read_epochs(file_path)
info = epochs.info
ch_names = epochs.ch_names


# Condtional Groups defintion

In [ ]:
# subject 25 removed due to Nan values in uncertanties
cfg = load_config()
groupby_dicts_list = {}
for i in cfg.dataset.test_subject_indices:
    if i == 25:
        continue
    remove_indices = []
    sigma2 = subjects_dicts[i]["uncertainties"]
    # throw away top 5% of values, hoping to remove outliers
    sigma2_indices = sigma2<=np.percentile(sigma2, 95)
    outlier_indices = ~sigma2_indices
    #hist_values.append(sigma2[i][vars_lower_x_percent_bool])   

    #This way the treshholds are calculated without the outliers
    low_threshold = np.percentile(sigma2[sigma2_indices], 33)
    high_threshold = np.percentile(sigma2[sigma2_indices], 66)
    # Treshholds are now defined without consideration of outliers.
    # BUT: outlier trials not removed from dataset!

    # Remove outliers by getting boolean array of them and using xor(^) with group_indices
    low_indices = (sigma2 < low_threshold)
    medium_indices = ((low_threshold<=sigma2) & (sigma2<=high_threshold))
    high_indices = (sigma2>high_threshold)^outlier_indices
    uncertainty_levels = {"low":low_indices, "medium":medium_indices, "high":high_indices}
    
    binary_label_fixed_zero_bool = subjects_dicts[i]["pred_binary_label_fixed"] == 0
    binary_label_fixed_one_bool = subjects_dicts[i]["pred_binary_label_fixed"] ==  1
    binary_label_fixed_dict = {"pred_zero" : binary_label_fixed_zero_bool, "pred_one" : binary_label_fixed_one_bool}


    
    groupby_dicts_list[i] = (uncertainty_levels,binary_label_fixed_dict)

    

# plot grouped topomaps

In [ ]:
subject_indices

In [ ]:
subjects = cfg.dataset.test_subject_indices
for si in subject_indices:
        get_and_plot_topomap_groupby(subjects_dicts[si]["explanations"], groupby_dicts_list[si][0], np.mean, info,  subject_idx=subject_indices[si], save_path="./groupby/Uncertainty/index_group/topoplots", axis=(0,2))

# define index group for every 100 trials. That allows us to inspet the change of importance across the continuous finetuning within a subject

In [ ]:
index_groups_all = {}
for i in range(len(subject_indices)):
    index_groups_subject = []
    start = 0
    end = 0
    while end < len(subjects_dicts[subject_indices[i]]["uncertainties"]):
        if len(subjects_dicts[subject_indices[i]]["uncertainties"]) >= end + 100:
            end += 100
        else:
            end = len(subjects_dicts[subject_indices[i]]["uncertainties"])

        subjects_dicts[subject_indices[i]]["uncertainties"][start:end]
        index_group = np.zeros(len(subjects_dicts[subject_indices[i]]["uncertainties"]), dtype=bool)
        index_group[start:end] = True
        index_groups_subject.append(index_group)
        start += 100
    index_groups_all[subject_indices[i]] = np.array(index_groups_subject)


In [ ]:
subjects = cfg.dataset.test_subject_indices
for si in subject_indices:
        get_and_plot_topomap_groupby(subjects_dicts[si]["explanations"], groupby_dicts_list[si][0], np.mean, info,  subject_idx=si, save_path="./groupby/Uncertainty/index_group/topoplots", axis=(0,2))

# timepoints analysis single subject and average over subjects

In [ ]:
def load_explanations_gradshap(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    return gradshap

In [ ]:
def load_explanations_saliency(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/saliency_explanations"

    saliency = np.load(os.path.join(load_path, f"saliency_explanations_subject_{subject_index}.npy"), allow_pickle=True)
    return saliency



In [ ]:
def load_ch_names(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    return ch_names

In [ ]:
def get_channel_importances(explanations, ch_names, take_abs=False):


    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial
    
    channel_importances = np.zeros(len(ch_names))

    for trial_idx, trial_normed in enumerate(explanations_normed):
        if take_abs:
            channel_importances += np.mean(np.abs(trial_normed), axis=1)
        else:
            channel_importances += np.mean(trial_normed, axis=1)
    
    channel_importances_dic = {ch_names[i]: channel_importances[i] for i in range(len(ch_names))}

    return channel_importances_dic

In [ ]:
def create_index_groups(n_samples, subject_index, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < n_samples:
        end = min(start + group_size, n_samples-20)
        
        index_group = np.zeros(n_samples, dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    

    return index_groups_subject

In [ ]:
def load_all_subject_results(subject_indices):
    results = {}
    for subject_index in subject_indices:
        results[subject_index]  = np.load(f'subject_{subject_index}_results.pkl', allow_pickle=True)
    return results

# List of subject indices to load
cfg = load_config()
subject_indices = cfg.dataset.test_subject_indices

# Load results for each subject
subjects_dicts = load_all_subject_results(subject_indices)

In [ ]:
def get_channel_importances_time_indexed(explanations, ch_names, index_groups, take_abs=False):
    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial_normed = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial_normed

    #channel_importances = np.zeros(( len(index_groups),len(ch_names)))
    channel_importances = []
    for i,index_group in enumerate(index_groups):
        explanations_index_group = explanations_normed[index_group]
        channel_importances_index_group = np.zeros(len(ch_names))
        
        for trial_idx, trial_normed in enumerate(explanations_index_group):
            if take_abs:
                # taking the mean within a trial is fine as different time poins are supposed to be more important than other
                channel_importances_index_group += np.mean(np.abs(trial_normed), axis=1)
            else:
                channel_importances_index_group += np.mean(trial_normed, axis=1)
    
        channel_importances_dic = {ch_names[i]: channel_importances_index_group[i] for i in range(len(ch_names))}
        channel_importances.append(channel_importances_dic)
    

    return channel_importances
    

In [ ]:
subjects_dicts[1]["explanations"].shape

In [ ]:
n = len(index_groups_all[2])
fig, axs = plt.subplots(nrows=1, ncols=n, figsize=(14,5))
for i in range(n):
    get_and_plot_topomap(np.abs(subjects_dicts[2]["explanations"]), np.mean, info, index_groups_all[2][i], subject_idx=subject_indices[0], save_path="./groupby/index_group/topoplots", axis=(0,2), ax = axs[i])
fig.savefig("subject_2_median_abs_importance_over_time.png")

In [ ]:
cwd = os.getcwd()
file_path = os.path.join(cwd, "../data/subject_007_preprocessed_combined_py.fif")
epochs = mne.read_epochs(file_path)
info = epochs.info
ch_names = epochs.ch_names

# We can now group by condtions and by the trials (i.e. plot for early stage of continuous finetuning vs later stage of continuous finetuning)


### But do we want to inspect how the index groups trials look for each subject (i.e. loop over index groups and then over subjects)
### or do we want to see how the importances develop over time for each subject (i.e. loop over subject and then over index group) 
### For now I Assume the second option is more important, as it allows us to see if continuous finetuning  helps/changes much

# groupby uncertainty

## aggregate over trial and timepoint dimension with mean

In [ ]:
subjects = cfg.dataset.test_subject_indices
for idx, si in enumerate(subject_indices):
    for j in range(len(index_groups_all[si])):
        get_and_plot_topomap_groupby(subjects_dicts[si]["explanations"], groupby_dicts_list[si][0], np.mean, info, indices=index_groups_all[si][j], subject_idx=si, save_path="./groupby/Uncertainty/index_group/topoplots", axis=(0,2))

## aggregate over timepoint dimension with mean, and over trial dimension with top-k

### linear update function

The differenence between the two approaches (mean vs top-k) is that when we aggregate over the trial timension with top-k, we work the assumption into the aggregation that every trial is of equal importance to the result.

In every trial get the top k(=20) channels. Using the update function update_channel_points_linear we accumulate channel importances across trials. The top channel gets k points, the second channel k-1 points i.e. points awared are decreased linearly, Until a minimum of 0.

In [ ]:
subjects_dicts[subject_indices[1]]["explanations"].shape

In [ ]:
groupby_dicts_list[1][0]["low"].shape

In [ ]:
points_progression_per_subject_uncertainty = []
for si in subject_indices:
    for j in range(len(index_groups_all[si])):
        points_progression_per_subject_uncertainty.append(top_k_groupby(subjects_dicts[si]["explanations"], groupby_dicts_list[si][0], ch_names, 10, update_channel_points_linear, index_group=index_groups_all[si][j]))

In [ ]:
points_progression_per_subject_uncertainty_abs =    []
for si in subject_indices:
    for j in range(len(index_groups_all[si])):
        points_progression_per_subject_uncertainty_abs.append((top_k_groupby(np.abs(subjects_dicts[si]["explanations"]), groupby_dicts_list[si][0], ch_names, 10, update_channel_points_same, index_group=index_groups_all[si][j])))

In [ ]:
points_progression_subject_2_abs = []
for j in range(len(index_groups_all[2])):
    points_progression_subject_2_abs.append(top_k(np.abs(subjects_dicts[2]["explanations"]), ch_names, 10, update_channel_points_linear, index_group=index_groups_all[2][j]))

In [ ]:
points_progression_subject_2 = []
for j in range(len(index_groups_all[2])):
    points_progression_subject_2.append(top_k((subjects_dicts[2]["explanations"]), ch_names, 10, update_channel_points_linear, index_group=index_groups_all[2][j]))

code up one version that just takes accumulated points index(for linear update funciont) and one that accumulates over trial dimension( for same update function)

In [ ]:
def top_k_channel_importances_linear_topoplot(point_progression_dict, info, groupby_name="uncertainty", index_group=np.array([]), subject_idx=2,save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    
    ncols = len(point_progression_dict.keys())
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(6,2))
    fig.suptitle(f"subject: {subject_idx} , indices: {np.where(index_group==True)[0][0]} : {(np.where(index_group==True)[0][-1])}")
    for idx, groupby_key in enumerate(point_progression_dict.keys()):
        if point_progression_dict[groupby_key]:
            channel_importancs = list(point_progression_dict[groupby_key][-1].values())
        else:
            continue
        if index_group.any() == False:
            axs[idx].set_title(f"{groupby_name}: {groupby_key}", y=0.9)
        else:
            axs[idx].set_title(f"{groupby_name}: {groupby_key}", y = 0.9)
        
        mne.viz.plot_topomap(channel_importancs , info, axes= axs[idx], show=False)

        dir_path = f"{save_path}/linear"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/top_k_linear_subject_{subject_idx}_topoplot.png")
        


In [ ]:
def top_k_channel_importances_linear_topoplot_comparison(point_progression_dict1, point_progression_dict2, info, groupby_name="uncertainty", index_group=np.array([]), subject_idx=2, save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    ncols = len(point_progression_dict1.keys())
    fig, axs = plt.subplots(nrows=2, ncols=ncols, figsize=(6 * ncols, 4))
    fig.suptitle(f"subject: {subject_idx} , indices: {np.where(index_group==True)[0][0]} : {(np.where(index_group==True)[0][-1])}")

    for idx, groupby_key in enumerate(point_progression_dict1.keys()):
        if point_progression_dict1[groupby_key]:
            channel_importancs1 = list(point_progression_dict1[groupby_key][-1].values())
        else:
            continue
        if point_progression_dict2[groupby_key]:
            channel_importancs2 = list(point_progression_dict2[groupby_key][-1].values())
        else:
            continue

        axs[0, idx].set_title(f"{groupby_name} 1: {groupby_key}", y=0.9)
        mne.viz.plot_topomap(channel_importancs1, info, axes=axs[0, idx], show=False)

        axs[1, idx].set_title(f"{groupby_name} 2: {groupby_key}", y=0.9)
        mne.viz.plot_topomap(channel_importancs2, info, axes=axs[1, idx], show=False)

    dir_path = f"{save_path}/linear_comparison"
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/top_k_linear_comparison_subject_{subject_idx}_topoplot.png")
        

In [ ]:
def top_k_channel_importances_linear_topoplot_noGroup(point_progression_list, info, index_group=np.array([]), subject_idx=2, save_path="refactor_test/groupby/uncertainty/topoplots/", ax=None, **kwargs):
    
    #if ax is None:
    #    ncols = len(point_progression_list)
    #    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(6 * ncols, 8))
    #    fig.suptitle(f"subject: {subject_idx} , indices: {np.where(index_group==True)[0][0]} : {(np.where(index_group==True)[0][-1])}")
    #else:
    #    axs = ax

    
    channel_importancs = np.array(list(point_progression_list[-1].values()))
  
        #if index_group.any() == False:
        #    axs[idx].set_title(f"Group: {idx}", y=0.9)
        #else:
        #    axs[idx].set_title(f"Group: {idx}", y=0.9)
        
    mne.viz.plot_topomap(channel_importancs, info, axes=ax, show=False)

    dir_path = f"{save_path}/linear"
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    if ax is None:
        fig.savefig(f"{dir_path}/top_k_linear_subject_{subject_idx}_topoplot.png")
        


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=len(index_groups_all[2]), figsize=(16,8))
for j in range(len(index_groups_all[2])):
    top_k_channel_importances_linear_topoplot_noGroup(points_progression_subject_2[j], info, subject_idx=2, index_group=index_groups_all[2][j], save_path="./groupby/index_group/topoplots", ax=axs[j])

fig.subplots_adjust(wspace=0.08)
fig.savefig("subject_2_topk_abs_importance_over_time.png")

     

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=len(index_groups_all[2]), figsize=(16,6))
for j in range(len(index_groups_all[2])):
    top_k_channel_importances_linear_topoplot_noGroup(points_progression_subject_2_abs[j], info, subject_idx=2, index_group=index_groups_all[2][j], save_path="./groupby/index_group/topoplots", ax=axs[j])

fig.subplots_adjust(wspace=0.08)
fig.savefig("subject_2_topk_abs_importance_over_time.png")


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=len(index_groups_all[2]), figsize=(16,3))
fig.tight_layout()
for j in range(len(index_groups_all[2])):
    top_k_channel_importances_linear_topoplot_noGroup(points_progression_subject_2_abs[j], info, subject_idx=2, index_group=index_groups_all[2][j], save_path="./groupby/index_group/topoplots", ax=axs[j])

fig.subplots_adjust(wspace=0.08)
fig.savefig("subject_2_topk_abs_importance_over_time.png")


In [ ]:
def channel_importance(explanations, ch_names, index_group=np.array([])):
    # explanation are of shape (trials, channels, timepoints)
    # we want to get the importance of each channel over all trials and timepoints
    # we also want each trial to be weighed equally so we MinMaxscale over timepoint and trial dimensions

    num_trials, num_channels, num_timepoints = explanations.shape
    channel_importance_scores = np.zeros(num_channels)
    channel_importance_dict = {}
    
    # Process each trial individually to ensure equal weighting
    for trial_idx in range(num_trials):
        trial_data = explanations[trial_idx]
        
        # Flatten channel and timepoint dimensions for scaling
        flattened = trial_data.reshape(-1)
        
        # Apply z-score normalization to the trial data
        mean = np.mean(flattened,)
        std = np.std(flattened)
        
        # Handle case where standard deviation is zero

        
        # Normalize the data
        normalized = (flattened - mean) / std
        trial_normalized = normalized.reshape(num_channels, num_timepoints)
        # Average across timepoints to get importance per channel for this trial
        trial_importance = np.mean(trial_normalized, axis=1)
        
        # Add to the cumulative scores
        channel_importance_scores += trial_importance
    
    # Average across all trials to get final importance scores
    channel_importance_scores /= num_trials

    for ch in ch_names:
        channel_importance_dict[ch] = channel_importance_scores[ch_names.index(ch)]

    
    return channel_importance_dict

In [ ]:
all_points_progressions = {}
for i,si in enumerate(subject_indices):
    subject_list = []
    for j in range(len(index_groups_all[si])):
        subject_list.append(top_k(np.abs(subjects_dicts[si]["explanations"]), ch_names, 10, update_channel_points_linear, index_group=index_groups_all[si][j]))
    all_points_progressions[si] = subject_list

# Find the minimum length of the lists within all_points_progressions
min_length = min(len(subject_list) for subject_list in all_points_progressions.values())

# Trim all lists within all_points_progressions to the minimum length
for subject_index in all_points_progressions:
    all_points_progressions[subject_index] = all_points_progressions[subject_index][:min_length]




In [ ]:
def aggregate_points_over_subjects(all_points_progressions, ch_names):
    aggregated_points = {}
    for subject_index in all_points_progressions:
        for i, points_progression in enumerate(all_points_progressions[subject_index]):
            if i not in aggregated_points:
                aggregated_points[i] = {ch: 0 for ch in ch_names}
            for ch in ch_names:
                aggregated_points[i][ch] += points_progression[-1][ch]
    return aggregated_points

# Aggregate points over subjects
aggregated_points = aggregate_points_over_subjects(all_points_progressions, ch_names)

# Print the aggregated points for verification
fig, axs = plt.subplots(nrows=1, ncols=len(aggregated_points), figsize=(16, 5))
for i in range(len(aggregated_points)):
    channel_importancs = np.array(list(aggregated_points[i].values()))
    mne.viz.plot_topomap(channel_importancs, info, axes=axs[i], show=False)
fig.subplots_adjust(wspace=0.08)
fig.savefig("average_topk_all_subjects.png")


observable that the further finetuning proceeds the more variance is introduced in the data

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot(points_progression_per_subject_uncertainty[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], save_path="./groupby/Uncertainty/index_group/topoplots")
        s+=1

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot(points_progression_per_subject_uncertainty_abs[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], save_path="./groupby/Uncertainty/index_group/topoplots")
        s+=1

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot_comparison(points_progression_per_subject_uncertainty[s], points_progression_per_subject_uncertainty_abs[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], save_path="./groupby/Uncertainty/index_group/topoplots")
        s+=1

#### Note here that subjects that have reasonale important channels (subjects 2,7,9 and maybe 11) the importance of the most important channel is more pronounced for the low and medium uncertainty trials.

### same update function

In [ ]:
points_progression_per_subject_uncertainty = []
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_func=update_channel_points_same, top_k_func=get_top_k_weighted_individual, index_group=index_groups_all[i][j])))

In [ ]:
def top_k_channel_importances_same_topoplot(point_progression_dict, index, info, groupby_name="uncertainty", subject_idx=2, index_group=np.array([]), save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    ncols = len(point_progression_dict[index].keys())
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(12,4))
    if index_group.any() == True:
        fig.suptitle(f"subject: {subject_idx}, indices: {np.where(index_group==True)[0][0]} : {(np.where(index_group==True)[0][-1])}")
    else:
        fig.suptitle(f"subject: {subject_idx}")

    for idx, groupby_key in enumerate(point_progression_dict[index].keys()):
        # subject_idx x groupby_key x all channel values x trial
        # trial set to 0, as all trials contain the same number of channels anyway
       
        channel_importancs = np.array(list(format_point_progression(point_progression_dict[index][groupby_key], ch_names).values())).sum(axis=1)
        len(channel_importancs)
        axs[idx].set_title(f"{groupby_name}: {groupby_key}")
        mne.viz.plot_topomap(channel_importancs , info, axes= axs[idx], show=False)

        dir_path = f"{save_path}/same"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/top_k_same_subject_{subject_idx}_topoplot.png")
        


In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_same_topoplot(points_progression_per_subject_uncertainty, s, info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], save_path="./groupby/Uncertainty/index_group/topoplots")
        s+=1

## General observation.
Using top-k instead of mean to aggreagte, the high uncertainty trials become a lof more noisy.\
That may suggest that the high uncertainty group is dominated by some trials when we aggregate over time_point and trial dimension with mean only.\
However, even after the removal of the trials with extremely high uncertainty, it does not seem like the regions of interest (on the motorcortex) are more important to the model according to the used XAI methods.

# grouby pred fixed

## aggregate over trial and timepoint dimension with mean

In [ ]:
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        get_and_plot_topomap_groupby(subjects_dicts[subject_indices[i]]["explanations"], groupby_dicts_list[i][1], np.mean, info, indices=index_groups_all[i][j], subject_idx=subject_indices[i], save_path="./groupby/Prediction_fixed/index_group/topoplots", groupby_name="pred_fixed", axis=(0,2))

## aggregate over timepoint dimension with mean, and over trial dimension with top-k

### linear update function

In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[subject_indices[i]]["explanations"], groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear, index_group=index_groups_all[i][j])))

In [ ]:
points_progression_per_subject_pred_fixed_abs = []
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        points_progression_per_subject_pred_fixed_abs.append((top_k_groupby(np.abs(subjects_dicts[subject_indices[i]]["explanations"]), groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear, index_group=index_groups_all[i][j])))

In [ ]:
s = 0
for idx, si in enumerate(subject_indices):
    for j in range(len(index_groups_all[si])):
        top_k_channel_importances_linear_topoplot_comparison(points_progression_per_subject_pred_fixed[s],points_progression_per_subject_pred_fixed_abs[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], groupby_name="pred_fixed",save_path="./groupby/Prediction_fixed/index_group/topoplots")
        s+=1

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot(points_progression_per_subject_pred_fixed[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], groupby_name="pred_fixed",save_path="./groupby/Prediction_fixed/index_group/topoplots")
        s+=1

#### Of interest here is that both contralateral sides of (presumably) the most reasonable subjects (2 and 7) one the motor cortex are marked as important here for the "prediction: one" condition compared to the "prediction: zero" condtion

#### Also note that i general, possibly noisy trials can be detected, subject 11 for example shows an obvious eye-blink artifact. Subjects 4, 22 and 24 may also result from noise but I am way less sure in these case.

### same update function

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_same_topoplot(points_progression_per_subject_pred_fixed,s, info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], groupby_name="pred_fixed",save_path="./groupby/Prediction_fixed/index_group/topoplots")
        s+=1

# grouby pred rolling

## mean

In [ ]:
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][2], "dl_agg", np.mean, info, indices=index_groups_all[i][j] ,subject_idx=subject_indices[i], groupby_name="pred_rolling", save_path="./groupby/Prediction_Fixed/index_group/topoplots", axis=(0,2))

## linear

In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_linear, index_group=index_groups_all[i][j])))

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot(points_progression_per_subject_pred_rolling[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j],groupby_name="pred_rolling",save_path="./groupby/Prediction_rolling/index_group/topoplots")
        s+=1

## same

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_same_topoplot(points_progression_per_subject_pred_rolling,s, info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], groupby_name="pred_rolling",save_path="./groupby/Prediction_rolling/index_group/topoplots")
        s+=1

# groupby true label

## mean

In [ ]:
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][3], "dl_agg", np.mean, info, indices = index_groups_all[i][j], subject_idx=subject_indices[i], groupby_name="true", save_path="./groupby/True_label/index_group/topoplots", axis=(0,2))

## linear

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_linear, index_group=index_groups_all[i][j])))

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_linear_topoplot(points_progression_per_subject_true[s], info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j] ,groupby_name="true",save_path="./groupby/True_label/index_group/topoplots")
        s+=1

## same

In [ ]:
s = 0
for i in range(len(subject_indices)):
    for j in range(len(index_groups_all[i])):
        top_k_channel_importances_same_topoplot(points_progression_per_subject_true,s, info, subject_idx=subject_indices[i], index_group=index_groups_all[i][j], groupby_name="true",save_path="./groupby/True_label/index_group/topoplots")
        s+=1

ToDo: Just like for distribution of uncertanties/variancs, check distributions of the raw labels fixed/rolling too. Possibly there are outliers too. If yes, would they have to be removed too